# Análisis temporal de Bitcoin — `CBBTCUSD (1).csv`

Este notebook adapta el reto de portafolio a una serie temporal diaria del precio de Bitcoin en USD. Cubre preparación (E0), EDA (E1), ingeniería de características, modelamiento temporal básico e interpretación (E2).

> Coloca `CBBTCUSD (1).csv` junto al notebook o en `/content/`. La base debe contener `observation_date` y `CBBTCUSD`.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown
from sklearn.metrics import mean_absolute_error, mean_squared_error
sns.set_theme(style='whitegrid')
pd.set_option('display.float_format', lambda x: f'{x:,.4f}')

## E0 — Preparación, niveles de medición y calidad

`observation_date` es una variable temporal/ordinal; `CBBTCUSD` es una variable cuantitativa continua de razón. Los rendimientos permiten comparar cambios relativos cuando el nivel del precio cambia mucho. No se imputan faltantes silenciosamente.

In [ ]:
def encontrar_archivo(nombre='CBBTCUSD (1).csv'):
    candidatos=[Path(nombre),Path('Capitulo4/Quiz')/nombre,Path('/content')/nombre,Path('../')/nombre]
    for p in candidatos:
        if p.exists(): return p
    encontrados=list(Path('.').rglob(nombre))
    return encontrados[0] if encontrados else None

ruta=encontrar_archivo()
if ruta is None:
    raise FileNotFoundError('No se encontró CBBTCUSD (1).csv. Colócalo junto al notebook o en /content/.')
df=pd.read_csv(ruta)
requeridas={'observation_date','CBBTCUSD'}
if not requeridas.issubset(df.columns): raise ValueError(f'Faltan columnas: {requeridas-set(df.columns)}')
df=df[['observation_date','CBBTCUSD']].copy()
df['observation_date']=pd.to_datetime(df['observation_date'],errors='coerce')
df['CBBTCUSD']=pd.to_numeric(df['CBBTCUSD'],errors='coerce')
df=df.sort_values('observation_date').reset_index(drop=True)
display(df.head()); display(df.tail())
print(f'Archivo: {ruta} | filas: {len(df):,} | periodo: {df.observation_date.min().date()} a {df.observation_date.max().date()}')

In [ ]:
calidad=pd.Series({'filas':len(df),'fechas_invalidas':df.observation_date.isna().sum(),'precios_faltantes':df.CBBTCUSD.isna().sum(),'fechas_duplicadas':df.observation_date.duplicated().sum(),'precios_no_positivos':(df.CBBTCUSD<=0).sum()})
display(calidad.to_frame('resultado'))
df['dias_entre_observaciones']=df.observation_date.diff().dt.days
huecos=df.loc[df.dias_entre_observaciones>1,['observation_date','dias_entre_observaciones']]
print(f'Huecos temporales: {len(huecos)}'); display(huecos.head(10))
df=df.dropna(subset=['observation_date','CBBTCUSD']).copy()
df=df[df.CBBTCUSD>0].copy()
assert df.observation_date.is_monotonic_increasing and (df.CBBTCUSD>0).all()

## Ingeniería de características

Se calculan cambios, rendimientos, variables calendario, medias móviles, volatilidad, máximos acumulados y drawdown. Los `NaN` iniciales de ventanas y rendimientos se conservan explícitamente.

In [ ]:
df['cambio_absoluto']=df.CBBTCUSD.diff()
df['rendimiento_diario']=df.CBBTCUSD.pct_change()
df['rendimiento_log']=np.log(df.CBBTCUSD).diff()
df['año']=df.observation_date.dt.year; df['mes']=df.observation_date.dt.month
df['trimestre']=df.observation_date.dt.to_period('Q').astype(str); df['dia_semana']=df.observation_date.dt.day_name()
df['maximo_acumulado']=df.CBBTCUSD.cummax()
df['drawdown_pct']=(df.CBBTCUSD/df.maximo_acumulado-1)*100
for ventana in [7,30,90]: df[f'media_movil_{ventana}d']=df.CBBTCUSD.rolling(ventana,min_periods=ventana).mean()
df['volatilidad_movil_30d']=df.rendimiento_diario.rolling(30,min_periods=20).std()*np.sqrt(365)
df['subida']=(df.rendimiento_diario>0).astype('Int64'); df['caida']=(df.rendimiento_diario<0).astype('Int64')
display(df.tail())

## E1 — EDA y comparación temporal

In [ ]:
primera,ultima=df.iloc[0],df.iloc[-1]
p0,p1=primera.CBBTCUSD,ultima.CBBTCUSD
cambio=p1-p0; variacion=(p1/p0-1)*100
imin,imax=df.CBBTCUSD.idxmin(),df.CBBTCUSD.idxmax()
r=df.rendimiento_diario.dropna()
resumen=pd.Series({'fecha_inicial':primera.observation_date.date(),'fecha_final':ultima.observation_date.date(),'precio_inicial_USD':p0,'precio_final_USD':p1,'cambio_absoluto_USD':cambio,'cambio_porcentual':variacion,'minimo_USD':df.loc[imin,'CBBTCUSD'],'fecha_minimo':df.loc[imin,'observation_date'].date(),'maximo_USD':df.loc[imax,'CBBTCUSD'],'fecha_maximo':df.loc[imax,'observation_date'].date(),'observaciones':len(df),'volatilidad_diaria':r.std(),'volatilidad_anualizada_365':r.std()*np.sqrt(365),'dias_positivos':(r>0).sum(),'dias_negativos':(r<0).sum(),'max_drawdown_pct':df.drawdown_pct.min()})
display(resumen.to_frame('valor'))
display(df[['CBBTCUSD','cambio_absoluto','rendimiento_diario','rendimiento_log']].describe())

In [ ]:
anual=df.set_index('observation_date').resample('YE').CBBTCUSD.agg(['first','last','min','max'])
anual['variacion_absoluta']=anual['last']-anual['first']; anual['variacion_porcentual']=(anual['last']/anual['first']-1)*100
mensual=df.set_index('observation_date').CBBTCUSD.resample('ME').agg(['first','last']); mensual['variacion_porcentual']=(mensual['last']/mensual['first']-1)*100; mensual['año']=mensual.index.year; mensual['mes']=mensual.index.month
display(anual); display(mensual.tail(12))
print('Mejores días'); display(df.nlargest(10,'rendimiento_diario')[['observation_date','CBBTCUSD','rendimiento_diario']])
print('Peores días'); display(df.nsmallest(10,'rendimiento_diario')[['observation_date','CBBTCUSD','rendimiento_diario']])

In [ ]:
fig,ax=plt.subplots(2,1,figsize=(15,10),sharex=True)
ax[0].plot(df.observation_date,df.CBBTCUSD,label='BTC/USD',color='darkorange')
for v in [7,30,90]: ax[0].plot(df.observation_date,df[f'media_movil_{v}d'],label=f'MM {v} días')
ax[0].set_title('Precio y medias móviles'); ax[0].set_ylabel('USD'); ax[0].legend()
ax[1].plot(df.observation_date,df.CBBTCUSD,color='navy'); ax[1].set_yscale('log'); ax[1].set_title('Precio en escala logarítmica'); ax[1].set_ylabel('USD (log)'); plt.tight_layout(); plt.show()
fig,ax=plt.subplots(2,2,figsize=(15,9))
ax[0,0].plot(df.observation_date,df.rendimiento_diario); ax[0,0].axhline(0,color='black'); ax[0,0].set_title('Rendimiento diario')
sns.histplot(r,bins=100,kde=True,ax=ax[0,1]); ax[0,1].set_title('Distribución de rendimientos')
sns.boxplot(data=df,x='año',y='rendimiento_diario',ax=ax[1,0]); ax[1,0].tick_params(axis='x',rotation=90); ax[1,0].set_title('Rendimientos por año')
ax[1,1].plot(df.observation_date,df.volatilidad_movil_30d,color='crimson'); ax[1,1].set_title('Volatilidad móvil anualizada'); plt.tight_layout(); plt.show()

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(16,5))
ax[0].fill_between(df.observation_date,df.drawdown_pct,0,color='crimson',alpha=.5); ax[0].set_title('Drawdown'); ax[0].set_ylabel('%')
heat=mensual.pivot(index='año',columns='mes',values='variacion_porcentual'); sns.heatmap(heat,cmap='RdYlGn',center=0,ax=ax[1]); ax[1].set_title('Variación mensual (%)'); plt.tight_layout(); plt.show()
plt.figure(figsize=(14,5)); colors=['seagreen' if x>=0 else 'firebrick' for x in anual.variacion_porcentual]; plt.bar(anual.index.year.astype(str),anual.variacion_porcentual,color=colors); plt.axhline(0,color='black'); plt.title('Variación anual'); plt.ylabel('%'); plt.xticks(rotation=90); plt.show()

## Modelamiento — baseline ingenuo

Se usa una división temporal 80/20 sin barajar. El modelo predice el siguiente precio con el último valor disponible. Es una referencia, no una prueba de causalidad ni una recomendación financiera.

In [ ]:
serie=df[['observation_date','CBBTCUSD']].dropna().reset_index(drop=True); corte=int(len(serie)*.8); train=serie.iloc[:corte]; test=serie.iloc[corte:].copy()
test['prediccion_naive']=train.CBBTCUSD.iloc[-1]
mae=mean_absolute_error(test.CBBTCUSD,test.prediccion_naive); rmse=np.sqrt(mean_squared_error(test.CBBTCUSD,test.prediccion_naive)); mape=np.mean(np.abs((test.CBBTCUSD-test.prediccion_naive)/test.CBBTCUSD))*100
display(pd.Series({'MAE_USD':mae,'RMSE_USD':rmse,'MAPE_pct':mape}).to_frame('baseline'))
plt.figure(figsize=(15,5)); plt.plot(test.observation_date,test.CBBTCUSD,label='real'); plt.plot(test.observation_date,test.prediccion_naive,label='baseline'); plt.legend(); plt.title('Validación temporal'); plt.show()

## E2 — Interpretación y conclusión

La interpretación se genera con los resultados calculados, evitando introducir cifras manualmente.

In [ ]:
mejor=anual.variacion_porcentual.idxmax().year; peor=anual.variacion_porcentual.idxmin().year
texto=f'''### Respuestas

Entre el **{primera.observation_date.date()}** y el **{ultima.observation_date.date()}**, Bitcoin **{'aumentó' if cambio>0 else 'disminuyó'}** de USD **{p0:,.2f}** a USD **{p1:,.2f}**, un cambio de USD **{cambio:,.2f}** ({variacion:.2f}%). El mínimo fue USD **{df.loc[imin,'CBBTCUSD']:,.2f}** el **{df.loc[imin,'observation_date'].date()}** y el máximo fue USD **{df.loc[imax,'CBBTCUSD']:,.2f}** el **{df.loc[imax,'observation_date'].date()}**.

El año con mayor variación positiva fue **{mejor}** y el de peor variación fue **{peor}**. La volatilidad anualizada estimada fue **{r.std()*np.sqrt(365):.2%}** y el máximo drawdown fue **{abs(df.drawdown_pct.min()):.2f}%**, lo que evidencia riesgo y fluctuaciones importantes.

**Conclusión:** la trayectoria de largo plazo debe interpretarse junto con rendimientos, volatilidad, drawdown y periodos intermedios; el precio inicial y final no describen por sí solos el riesgo. La base termina el **{ultima.observation_date.date()}**, así que no se infieren resultados posteriores.
'''; display(Markdown(texto))

## Rúbrica cubierta por el código

- **Preparación y transformación:** carga robusta, tipos, ordenamiento, faltantes, duplicados y continuidad temporal.
- **Niveles de medición:** explicación de fecha, precio y rendimientos.
- **EDA:** estadísticos, extremos, cambios, días positivos/negativos y análisis anual/mensual.
- **Ingeniería de características:** cambios, rendimientos simples/logarítmicos, calendario, medias móviles, volatilidad, drawdown y banderas.
- **Comparación temporal:** fechas inicial/final y periodos anuales/mensuales.
- **Visualización:** escala normal/log, tendencias, distribución, boxplot, volatilidad, drawdown, barras y heatmap.
- **Modelamiento:** baseline, partición temporal y MAE/RMSE/MAPE.
- **Interpretación y comunicación:** respuestas automáticas, conclusión, riesgos y limitaciones.

La participación en mesa redonda, la exposición oral, la argumentación y el trabajo colaborativo no pueden evaluarse únicamente desde el código.